# 03 — Version 2: Advanced Content-Based Recommendation

This notebook explores **Version 2: Advanced Content-Based Recommendation**.
- **Features:** Engineered textual representation combining:
  - Cleaned movie title
  - Normalized title (handling inverted articles like "The Matrix")
  - Genres tokens
  - Release year
  - Decade token (e.g. `1990s`)
- **Vectorization:** `TfidfVectorizer` (with unigram + bigram, sublinear TF scaling).
- **Similarity Metric:** Cosine similarity.


In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data_loader import load_movies
from src.preprocessing import preprocess_movies
from src.version2.advanced_content import AdvancedContentRecommender

movies_df = load_movies()
movies_clean = preprocess_movies(movies_df)

print("Sample engineered text feature:")
print(movies_clean[['DisplayTitle', 'V2_CombinedFeatures']].head(3))

# Fit Version 2
v2_model = AdvancedContentRecommender()
v2_model.fit(movies_clean)
print("\nTF-IDF Matrix Shape:", v2_model.tfidf_matrix.shape)


## Advanced Recommendations for Toy Story (MovieID = 1)


In [ ]:
recs_v2 = v2_model.recommend(movie_id=1, top_k=10)
recs_df = pd.DataFrame(recs_v2)
recs_df[['rank', 'title', 'genres', 'similarity_score']]


## Comparison: V1 (Genre Only) vs V2 (TF-IDF Metadata) for The Matrix


In [ ]:
from src.version1.basic_content import BasicContentRecommender
v1_model = BasicContentRecommender().fit(movies_clean)

v1_recs = v1_model.recommend(2571, top_k=5)
v2_recs = v2_model.recommend(2571, top_k=5)

print("=== V1: Genre-Only Cosine Similarity ===")
for r in v1_recs:
    print(f"{r['rank']}. {r['title']} | Score: {r['similarity_score']}")

print("\n=== V2: Engineered TF-IDF Metadata Cosine Similarity ===")
for r in v2_recs:
    print(f"{r['rank']}. {r['title']} | Score: {r['similarity_score']}")
